# 32. Machine Learning Pipelines & `ColumnTransformer`

A **Pipeline** chains data preprocessing transformers and a terminal estimator into a single unified object. This prevents **data leakage** during cross-validation and ensures transformations applied to training data are identically replicated on raw inference inputs.

### Learning Objectives:
- Construct multi-stage pipelines with `sklearn.pipeline.Pipeline`
- Handle mixed tabular types (numeric + categorical) simultaneously with `ColumnTransformer`
- Optimize transformers and estimators together in cross-validation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Heterogeneous customer dataset
df = pd.DataFrame({
    'Age': [25, 45, np.nan, 35, 52, 23, 40, 60],
    'Salary': [50000, 85000, 62000, np.nan, 110000, 48000, 95000, 130000],
    'Department': ['IT', 'HR', 'Sales', 'IT', 'Sales', 'HR', 'IT', 'Sales'],
    'Promoted': [0, 1, 0, 1, 1, 0, 1, 1]
})

X = df[['Age', 'Salary', 'Department']]
y = df['Promoted']

numeric_features = ['Age', 'Salary']
categorical_features = ['Department']

# Numeric Transformer Pipeline: Impute -> Scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Transformer Pipeline: Impute -> OneHotEncode
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Composite ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Full End-to-End Production Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42))
])

full_pipeline.fit(X, y)
print("Pipeline successfully fitted! Inference ready.")

# Predict directly on raw incoming input DataFrame!
new_employee = pd.DataFrame({'Age': [30], 'Salary': [75000], 'Department': ['IT']})
prediction = full_pipeline.predict(new_employee)
print(f"Prediction for new employee: {'Promoted' if prediction[0] == 1 else 'Not Promoted'}")

## 🎯 Exercises & Practice
1. Explain why using `Pipeline` eliminates data leakage during K-fold cross-validation.

In [ ]:
# Exercise Solution
# Answer: Inside cross-validation, the Pipeline re-executes fit_transform on each training fold independently and only transforms the validation fold, preventing any global mean/scale leakage from test splits.